# Tableau → Fabric: Migration Assessment — Play 5

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This notebook builds an **estate migration-assessment semantic model** over the Tableau
metadata that Play 2 landed. It derives a handful of tidy, analysis-ready `mig_*` Delta
tables, deploys a single DirectLake semantic model with a `_Metrics` table of
migration-KPI DAX measures, and is the data source for the deployable Power BI dashboard
template under `Play5/dashboard/`.

**Pipeline order:** Play 2 → **Play 5** (independent of Play 3/4)

```
Metadata_Lakehouse (Play 2 output)
  tableau_datasources / tableau_fields / tableau_lineage / tableau_workbooks
        ↓  (this notebook: derive + assess)
Metadata_Lakehouse (Play 5 derived)
  mig_datasources / mig_fields / mig_connections / mig_workbooks
  mig_dim_project / mig_dim_owner
        ↓  generate TMDL + deploy via Fabric REST
Fabric Workspace
  "Tableau Migration Assessment" semantic model (DirectLake → Metadata_Lakehouse)
  _Metrics table: Data Sources, Workbooks, Calculated Fields, Unique Connections,
  Flat-File Sources, Reused Data Sources, Stale Extracts, Stale Workbooks, ...
        ↓  connect the dashboard template
Play5/dashboard/  (deploy in Power BI Desktop / Fabric)
```

**Migration KPIs answered:** how many datasources & workbooks, how many distinct
connection instances and connector types, how many flat-file / one-off sources, how
many reused datasources, calculated-field density, certification coverage, and stale
extracts / stale workbooks.


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `METADATA_LAKEHOUSE` | Play 2 metadata lakehouse display name | e.g. `Metadata_Lakehouse` |
| `MODEL_DISPLAY_NAME` | Name for the deployed assessment model | e.g. `Tableau Migration Assessment` |
| `STALE_DAYS` | Days after which an extract / workbook is "stale" | default `90` |
| `FLAT_FILE_TYPES` | Tableau connectionTypes treated as flat-file / one-off | tunable set (see below) |
| `OVERWRITE` | Overwrite the model if it already exists | `True` to update, `False` to skip |

Attach this notebook to the **Metadata_Lakehouse** (the Play 2 output lakehouse) as its
default lakehouse before running.


In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
METADATA_LAKEHOUSE = "Metadata_Lakehouse"           # Play 2 output lakehouse (display name)
MODEL_DISPLAY_NAME = "Tableau Migration Assessment" # deployed semantic model name
STALE_DAYS         = 90                              # extract/workbook staleness threshold
OVERWRITE          = True                            # True = update existing model

# Tableau connectionType values treated as flat-file / one-off sources. Tune for your
# environment (leave as-is to use the library default).
FLAT_FILE_TYPES = {
    "excel-direct", "excel-reader", "textscan", "msaccess",
    "google-sheets", "googlesheets", "json", "jsonfile",
    "spatial", "shapefile", "cdata", "webdata-direct",
}

# ── FABRIC REST API ──────────────────────────────────────────────────────────
FABRIC_API = "https://api.fabric.microsoft.com/v1"

import requests, json, base64, uuid, re, time
from datetime import datetime, timezone
from pyspark.sql.types import (StructType, StructField, StringType,
                               BooleanType, LongType, TimestampType)

# Fabric token via managed identity
token = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Workspace ID from notebook context
WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")

# Resolve the metadata lakehouse ID by display name (DirectLake binds to THIS lakehouse,
# which holds both the raw Play 2 tables and the derived mig_* tables this notebook writes)
_lh = requests.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/lakehouses", headers=HEADERS)
_lh.raise_for_status()
METADATA_LAKEHOUSE_ID = next(
    (l["id"] for l in _lh.json().get("value", []) if l["displayName"] == METADATA_LAKEHOUSE), None)
if not METADATA_LAKEHOUSE_ID:
    raise ValueError(f"Lakehouse '{METADATA_LAKEHOUSE}' not found in workspace {WORKSPACE_ID}")

DIRECTLAKE_URL         = f"https://onelake.dfs.fabric.microsoft.com/{WORKSPACE_ID}/{METADATA_LAKEHOUSE_ID}"
EXPRESSION_SOURCE_NAME = f"DirectLake - {METADATA_LAKEHOUSE}"

print(f"✓ Workspace:        {WORKSPACE_ID}")
print(f"✓ Metadata lakehouse: {METADATA_LAKEHOUSE} ({METADATA_LAKEHOUSE_ID})")
print(f"✓ DirectLake target:  {DIRECTLAKE_URL}")


## Assessment library

The transforms (raw Play 2 → tidy `mig_*`), the TMDL/DAX generators, and the
Delta-schema contract below are validated offline (82 assertions in
`play5_assess.py` / `play5_tests.py`). They are embedded here verbatim so the deployed
notebook matches the tested logic exactly. The config cell above overrides
`FLAT_FILE_TYPES` and `STALE_DAYS` at call time.


In [ ]:
"""
Play 5 — Tableau→Fabric Migration Assessment: estate semantic-model generator.

Offline-first build (mirrors Play 4). This module is PURE Python (no Spark/Fabric)
so it can be unit-validated standalone, then ported into the Play 5 notebook.

Two halves:
  1. TRANSFORMS — turn the raw Play 2 metadata tables into tidy, analysis-ready
     "mig_*" tables with the boolean/numeric flags the KPIs need.
  2. TMDL GENERATORS — emit a single DirectLake "estate" semantic model over those
     derived tables, plus a _Metrics table carrying the migration-KPI DAX measures.

The derived-table SCHEMAS are fixed here (we create the Delta tables ourselves in the
notebook), so DirectLake column dataTypes are known constants — no schema introspection.
"""
import re
import uuid
import json
import base64
from datetime import datetime, timezone

# ──────────────────────────────────────────────────────────────────────────────
# TUNABLES
# ──────────────────────────────────────────────────────────────────────────────
# A datasource/workbook is "stale" if its last extract refresh / last update is
# older than this many days. Tunable per estate.
DEFAULT_STALE_DAYS = 90

# Tableau connectionType values that represent flat-file / one-off sources (not a
# governed database). Lower-cased compare. Tunable — extend for your environment.
DEFAULT_FLAT_FILE_TYPES = frozenset({
    "excel-direct", "excel-reader", "textscan", "msaccess",
    "google-sheets", "googlesheets", "json", "jsonfile",
    "spatial", "shapefile", "cdata", "webdata-direct",
})


# ──────────────────────────────────────────────────────────────────────────────
# VALUE HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def _is_na(v):
    return v is None or (isinstance(v, float) and v != v)


def _truthy(v):
    if _is_na(v):
        return False
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes", "t")
    return bool(v)


def _to_int(v, default=0):
    if _is_na(v):
        return default
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return default


def _s(v):
    if _is_na(v):
        return None
    s = str(v).strip()
    return s or None


def parse_iso(v):
    """Parse a Tableau ISO-8601 timestamp (may end in 'Z') to a tz-aware UTC datetime.
    Returns None if blank/unparseable."""
    s = _s(v)
    if not s:
        return None
    s = s.replace("Z", "+00:00")
    try:
        dt = datetime.fromisoformat(s)
    except ValueError:
        # Fall back to date-only or space-separated forms.
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d"):
            try:
                dt = datetime.strptime(s[:19], fmt)
                break
            except ValueError:
                continue
        else:
            return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _days_since(v, now):
    dt = parse_iso(v)
    if dt is None:
        return None
    return (now - dt).days


# ──────────────────────────────────────────────────────────────────────────────
# TRANSFORMS  (raw Play 2 rows -> tidy mig_* rows). Operate on list-of-dicts so they
# are framework-agnostic; the notebook feeds df.to_dict("records") and writes the
# returned rows back to Delta.
# ──────────────────────────────────────────────────────────────────────────────
def build_mig_datasources(rows, now=None, stale_days=DEFAULT_STALE_DAYS):
    now = now or datetime.now(timezone.utc)
    out = []
    for r in rows:
        days = _days_since(r.get("extract_last_refresh"), now)
        has_extracts = _truthy(r.get("has_extracts"))
        reused = _to_int(r.get("downstream_workbook_count")) > 1
        # Refresh staleness applies ONLY to extract-based sources. Live datasources
        # have no refresh signal and are never "refresh-stale" (see Workbooks for
        # content staleness). days_since_refresh is nullable (None) — no -1 sentinel,
        # so AVERAGEX over it stays correct without special-casing.
        is_refresh_stale = bool(has_extracts and days is not None and days > stale_days)
        out.append({
            "datasource_id":            _s(r.get("datasource_id")),
            "name":                     _s(r.get("name")),
            "project_name":             _s(r.get("project_name")),
            "owner_name":               _s(r.get("owner_name")),
            "is_certified":             _truthy(r.get("is_certified")),
            "certification_status":     _s(r.get("certification_status")),
            "has_extracts":             has_extracts,
            "connection_summary":       _s(r.get("connection_types")),
            "downstream_workbook_count": _to_int(r.get("downstream_workbook_count")),
            "field_count":              _to_int(r.get("field_count")),
            "is_reused":                bool(reused),
            "extract_last_refresh":     parse_iso(r.get("extract_last_refresh")),
            "days_since_refresh":       days,  # None when no extract
            "is_refresh_stale":         is_refresh_stale,
        })
    return out


def build_mig_connections(lineage_rows, datasource_rows=None,
                         flat_file_types=DEFAULT_FLAT_FILE_TYPES):
    """Build one row per real upstream connection INSTANCE from the lineage table:
    (datasource_id, database_name, connection_type, is_flat_file). This is the true
    connection grain (e.g. 12 SQL Server databases on 3 servers = 12 instances), which
    lets the model count distinct *connections* AND distinct connection *types*.

    Falls back to exploding a datasource's comma-joined `connection_types` SET (with a
    null database_name) for any datasource that has NO upstream_table lineage rows, so
    estates without table-level lineage still get type-level coverage."""
    flat = {t.lower() for t in flat_file_types}
    out = []
    covered = set()
    seen = set()
    for r in lineage_rows:
        if _s(r.get("relationship_type")) != "upstream_table":
            continue
        ct = _s(r.get("connection_type"))
        db = _s(r.get("database_name"))
        ds_id = _s(r.get("datasource_id"))
        if ds_id is None or (ct is None and db is None):
            continue
        covered.add(ds_id)
        key = (ds_id, (db or "").lower(), (ct or "").lower())
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "datasource_id":   ds_id,
            "datasource_name": _s(r.get("datasource_name")),
            "database_name":   db,
            "connection_type": ct,
            "is_flat_file":    bool(ct and ct.lower() in flat),
        })

    # Fallback: datasources with no upstream_table lineage -> explode the type set.
    for r in (datasource_rows or []):
        ds_id = _s(r.get("datasource_id"))
        if ds_id is None or ds_id in covered:
            continue
        ds_name = _s(r.get("name"))
        raw = _s(r.get("connection_types")) or ""
        local = set()
        for part in raw.split(","):
            ct = part.strip()
            if not ct or ct.lower() in local:
                continue
            local.add(ct.lower())
            out.append({
                "datasource_id":   ds_id,
                "datasource_name": ds_name,
                "database_name":   None,
                "connection_type": ct,
                "is_flat_file":    ct.lower() in flat,
            })
    return out


def build_dim_project(datasource_rows, workbook_rows):
    """Distinct, non-null project names across datasources + workbooks. A shared
    dimension so a single project slicer cross-filters BOTH asset tables."""
    names = set()
    for r in datasource_rows:
        p = _s(r.get("project_name"))
        if p:
            names.add(p)
    for r in workbook_rows:
        p = _s(r.get("project_name"))
        if p:
            names.add(p)
    return [{"project_name": p} for p in sorted(names)]


def build_dim_owner(datasource_rows, workbook_rows):
    """Distinct, non-null owner names across datasources + workbooks (shared dim)."""
    names = set()
    for r in datasource_rows:
        o = _s(r.get("owner_name"))
        if o:
            names.add(o)
    for r in workbook_rows:
        o = _s(r.get("owner_name"))
        if o:
            names.add(o)
    return [{"owner_name": o} for o in sorted(names)]


def build_mig_fields(rows):
    out = []
    for r in rows:
        ft = _s(r.get("field_type"))
        out.append({
            "datasource_id":   _s(r.get("datasource_id")),
            "datasource_name": _s(r.get("datasource_name")),
            "field_name":      _s(r.get("field_name")),
            "field_type":      ft,
            "data_type":       _s(r.get("data_type")),
            "role":            _s(r.get("role")),
            "is_calculated":   ft == "CalculatedField",
            "is_hidden":       _truthy(r.get("is_hidden")),
        })
    return out


def build_mig_workbooks(rows, now=None, stale_days=DEFAULT_STALE_DAYS):
    now = now or datetime.now(timezone.utc)
    out = []
    for r in rows:
        days = _days_since(r.get("updated_at"), now)
        # Workbook content staleness: not updated within STALE_DAYS. Unknown update
        # date -> not flagged (conservative). days nullable (None), no sentinel.
        is_content_stale = bool(days is not None and days > stale_days)
        out.append({
            "workbook_id":                _s(r.get("workbook_id")),
            "name":                       _s(r.get("name")),
            "project_name":               _s(r.get("project_name")),
            "owner_name":                 _s(r.get("owner_name")),
            "sheet_count":                _to_int(r.get("sheet_count")),
            "published_datasource_count": _to_int(r.get("published_datasource_count")),
            "updated_at":                 parse_iso(r.get("updated_at")),
            "days_since_update":          days,  # None when update date unknown
            "is_content_stale":           is_content_stale,
        })
    return out


# ──────────────────────────────────────────────────────────────────────────────
# FIXED DERIVED-TABLE SCHEMAS  (display name, delta/entity name, columns)
# Each column: (name, tmdl_type, summarizeBy, isHidden). We create these Delta tables
# ourselves, so sourceColumn == name and dataType is authoritative.
# ──────────────────────────────────────────────────────────────────────────────
ESTATE_TABLES = [
    {
        "display": "Datasources",
        "entity":  "mig_datasources",
        "columns": [
            ("datasource_id", "string", "none", True),
            ("name", "string", "none", False),
            ("project_name", "string", "none", False),
            ("owner_name", "string", "none", False),
            ("is_certified", "boolean", "none", False),
            ("certification_status", "string", "none", False),
            ("has_extracts", "boolean", "none", False),
            ("connection_summary", "string", "none", False),
            ("downstream_workbook_count", "int64", "sum", False),
            ("field_count", "int64", "sum", False),
            ("is_reused", "boolean", "none", False),
            ("extract_last_refresh", "dateTime", "none", False),
            ("days_since_refresh", "int64", "none", False),
            ("is_refresh_stale", "boolean", "none", False),
        ],
    },
    {
        "display": "Fields",
        "entity":  "mig_fields",
        "columns": [
            ("datasource_id", "string", "none", True),
            ("datasource_name", "string", "none", False),
            ("field_name", "string", "none", False),
            ("field_type", "string", "none", False),
            ("data_type", "string", "none", False),
            ("role", "string", "none", False),
            ("is_calculated", "boolean", "none", False),
            ("is_hidden", "boolean", "none", False),
        ],
    },
    {
        "display": "Connections",
        "entity":  "mig_connections",
        "columns": [
            ("datasource_id", "string", "none", True),
            ("datasource_name", "string", "none", False),
            ("database_name", "string", "none", False),
            ("connection_type", "string", "none", False),
            ("is_flat_file", "boolean", "none", False),
        ],
    },
    {
        "display": "Workbooks",
        "entity":  "mig_workbooks",
        "columns": [
            ("workbook_id", "string", "none", True),
            ("name", "string", "none", False),
            ("project_name", "string", "none", False),
            ("owner_name", "string", "none", False),
            ("sheet_count", "int64", "sum", False),
            ("published_datasource_count", "int64", "sum", False),
            ("updated_at", "dateTime", "none", False),
            ("days_since_update", "int64", "none", False),
            ("is_content_stale", "boolean", "none", False),
        ],
    },
    {
        "display": "Project",
        "entity":  "mig_dim_project",
        "columns": [
            ("project_name", "string", "none", False),
        ],
    },
    {
        "display": "Owner",
        "entity":  "mig_dim_owner",
        "columns": [
            ("owner_name", "string", "none", False),
        ],
    },
]

# Relationships: child (many) -> dimension (one). Shared Project/Owner dims let one
# slicer cross-filter BOTH Datasources and Workbooks. datasource_id links the
# field/connection facts to the datasource dimension. Single cross-filter direction
# (the default): always slice from a dimension, never from a child fact.
ESTATE_RELATIONSHIPS = [
    {"from_table": "Fields", "from_col": "datasource_id",
     "to_table": "Datasources", "to_col": "datasource_id", "kind": "many_to_one"},
    {"from_table": "Connections", "from_col": "datasource_id",
     "to_table": "Datasources", "to_col": "datasource_id", "kind": "many_to_one"},
    {"from_table": "Datasources", "from_col": "project_name",
     "to_table": "Project", "to_col": "project_name", "kind": "many_to_one"},
    {"from_table": "Workbooks", "from_col": "project_name",
     "to_table": "Project", "to_col": "project_name", "kind": "many_to_one"},
    {"from_table": "Datasources", "from_col": "owner_name",
     "to_table": "Owner", "to_col": "owner_name", "kind": "many_to_one"},
    {"from_table": "Workbooks", "from_col": "owner_name",
     "to_table": "Owner", "to_col": "owner_name", "kind": "many_to_one"},
]

# Migration-KPI catalog: (measure name, DAX, formatString). Display names above are
# what these reference. Order is the display order in the _Metrics table.
# DISTINCTCOUNT measures guard against BLANK so an empty key isn't counted as a value.
ESTATE_MEASURES = [
    ("Data Sources", "COUNTROWS('Datasources')", "#,0"),
    ("Workbooks", "COUNTROWS('Workbooks')", "#,0"),
    ("Fields", "COUNTROWS('Fields')", "#,0"),
    ("Calculated Fields",
     "CALCULATE(COUNTROWS('Fields'), 'Fields'[is_calculated] = TRUE())", "#,0"),
    ("Unique Connections",
     "CALCULATE(DISTINCTCOUNT('Connections'[database_name]), NOT ISBLANK('Connections'[database_name]))", "#,0"),
    ("Unique Connection Types",
     "CALCULATE(DISTINCTCOUNT('Connections'[connection_type]), NOT ISBLANK('Connections'[connection_type]))", "#,0"),
    ("Flat-File Sources",
     "CALCULATE(DISTINCTCOUNT('Connections'[datasource_id]), 'Connections'[is_flat_file] = TRUE())", "#,0"),
    ("Flat-File One-Off Sources",
     "CALCULATE(DISTINCTCOUNT('Connections'[datasource_id]), 'Connections'[is_flat_file] = TRUE(), 'Datasources'[is_reused] = FALSE())", "#,0"),
    ("Reused Data Sources",
     "CALCULATE(COUNTROWS('Datasources'), 'Datasources'[is_reused] = TRUE())", "#,0"),
    ("Certified Data Sources",
     "CALCULATE(COUNTROWS('Datasources'), 'Datasources'[is_certified] = TRUE())", "#,0"),
    ("Extract-Based Sources",
     "CALCULATE(COUNTROWS('Datasources'), 'Datasources'[has_extracts] = TRUE())", "#,0"),
    ("Stale Extracts",
     "CALCULATE(COUNTROWS('Datasources'), 'Datasources'[is_refresh_stale] = TRUE())", "#,0"),
    ("Stale Workbooks",
     "CALCULATE(COUNTROWS('Workbooks'), 'Workbooks'[is_content_stale] = TRUE())", "#,0"),
    ("Hidden Fields",
     "CALCULATE(COUNTROWS('Fields'), 'Fields'[is_hidden] = TRUE())", "#,0"),
    ("Avg Extract Age (days)",
     "AVERAGEX(FILTER('Datasources', 'Datasources'[has_extracts] = TRUE()), 'Datasources'[days_since_refresh])", "#,0.0"),
    ("Avg Fields per Source",
     "DIVIDE(COUNTROWS('Fields'), COUNTROWS('Datasources'))", "#,0.0"),
    ("Calc Field Density",
     "DIVIDE([Calculated Fields], COUNTROWS('Datasources'))", "#,0.00"),
    ("Certification Rate",
     "DIVIDE([Certified Data Sources], [Data Sources])", "0.0%"),
]


# Delta-schema contract: the Spark simpleString type each mig_* column MUST land as.
# The notebook asserts the written Delta schema against this BEFORE generating TMDL,
# so a pandas object/string drift can't silently break DirectLake binding (Play 4 bug).
_TMDL_TO_SPARK = {"string": {"string"}, "boolean": {"boolean"},
                  "int64": {"int", "integer", "bigint", "long", "short", "byte"},
                  "dateTime": {"timestamp", "timestamp_ntz", "date"}}


def expected_delta_contract():
    """{entity_name: {column_name: set(acceptable spark simpleString types)}}."""
    contract = {}
    for t in ESTATE_TABLES:
        contract[t["entity"]] = {
            c[0]: set(_TMDL_TO_SPARK[c[1]]) for c in t["columns"]}
    return contract


def check_delta_schema(entity, actual_fields):
    """actual_fields: list of (column_name, spark_simpleString). Returns list of
    human-readable mismatch strings (empty == OK). Use in the notebook to fail fast."""
    contract = expected_delta_contract().get(entity, {})
    actual = {n: (t or "").lower().split("(")[0] for n, t in actual_fields}
    problems = []
    for col, ok_types in contract.items():
        if col not in actual:
            problems.append(f"{entity}.{col}: MISSING from Delta table")
        elif actual[col] not in ok_types:
            problems.append(
                f"{entity}.{col}: Delta type '{actual[col]}' not in {sorted(ok_types)}")
    return problems


# ──────────────────────────────────────────────────────────────────────────────
# TMDL GENERATORS  (shared identifier quoting + column/table/model emitters)
# ──────────────────────────────────────────────────────────────────────────────
_UNQUOTED = re.compile(r"^[A-Za-z_][A-Za-z0-9_\-]*$")


def q(name):
    if _UNQUOTED.match(name):
        return name
    return "'" + name.replace("'", "''") + "'"


def _format_string(tmdl_type, summarize):
    if tmdl_type == "dateTime":
        return "Short Date"
    if tmdl_type == "int64":
        return "#,0"
    if tmdl_type in ("double", "decimal") and summarize == "sum":
        return "#,0.00"
    return None


def generate_column_tmdl(col_name, tmdl_type, summarize, is_hidden):
    lines = [f"\tcolumn {q(col_name)}", f"\t\tdataType: {tmdl_type}"]
    if is_hidden:
        lines.append("\t\tisHidden")
    fmt = _format_string(tmdl_type, summarize)
    if fmt:
        lines.append(f"\t\tformatString: {fmt}")
    lines.append(f"\t\tlineageTag: {uuid.uuid4()}")
    lines.append(f"\t\tsourceLineageTag: {col_name}")
    lines.append(f"\t\tsummarizeBy: {summarize}")
    lines.append(f"\t\tsourceColumn: {col_name}")
    lines.append("")
    lines.append("\t\tannotation SummarizationSetBy = Automatic")
    return "\n" + "\n".join(lines) + "\n"


def generate_table_tmdl(table_display_name, delta_table_name, columns_tmdl, expression_source):
    return (
        f"table {q(table_display_name)}\n"
        f"\tlineageTag: {uuid.uuid4()}\n"
        f"\tsourceLineageTag: [dbo].[{delta_table_name}]\n"
        f"{columns_tmdl}\n"
        f"\tpartition {delta_table_name} = entity\n"
        f"\t\tmode: directLake\n"
        f"\t\tsource\n"
        f"\t\t\tentityName: {delta_table_name}\n"
        f"\t\t\tschemaName: dbo\n"
        f"\t\t\texpressionSource: {q(expression_source)}\n\n"
    )


def generate_metric_tmdl(name, dax, format_string):
    """A real DAX measure (not a stub) for the _Metrics table."""
    lines = [f"\n\tmeasure {q(name)} = {dax}"]
    if format_string:
        lines.append(f"\t\tformatString: {format_string}")
    lines.append(f"\t\tlineageTag: {uuid.uuid4()}")
    lines.append("\t\tannotation SummarizationSetBy = Automatic")
    return "\n".join(lines) + "\n"


def generate_metrics_table_tmdl(measures):
    """Canonical measures-holder: a single-row calculated table (no Delta binding)
    whose only job is to host the migration-KPI measures."""
    column = (
        "\n\tcolumn Value\n"
        "\t\tdataType: string\n"
        "\t\tisHidden\n"
        f"\t\tlineageTag: {uuid.uuid4()}\n"
        "\t\tsummarizeBy: none\n"
        "\t\tsourceColumn: [Value]\n"
        "\t\ttype: calculatedTableColumn\n"
    )
    measures_tmdl = "".join(
        generate_metric_tmdl(n, dax, fmt) for (n, dax, fmt) in measures)
    partition = (
        "\tpartition _Metrics = calculated\n"
        "\t\tmode: import\n"
        '\t\tsource = Row("Value", BLANK())\n'
    )
    return (
        f"table _Metrics\n"
        f"\tlineageTag: {uuid.uuid4()}\n"
        f"{column}"
        f"{measures_tmdl}\n"
        f"{partition}\n"
        f"\tannotation PBI_Id = _Metrics\n\n"
    )


def generate_relationships_tmdl(rels):
    if not rels:
        return None
    blocks = []
    for r in rels:
        blocks.append("\n".join([
            f"relationship {uuid.uuid4()}",
            f"\tfromColumn: {q(r['from_table'])}.{q(r['from_col'])}",
            f"\ttoColumn: {q(r['to_table'])}.{q(r['to_col'])}",
        ]))
    return "\n\n".join(blocks) + "\n"


def generate_expressions_tmdl(expression_name, directlake_url):
    return (
        f"expression {q(expression_name)} =\n"
        f"\t\tlet\n"
        f'\t\t    Source = AzureStorage.DataLake("{directlake_url}", [HierarchicalNavigation=true])\n'
        f"\t\tin\n"
        f"\t\t    Source\n"
        f"\tlineageTag: {uuid.uuid4()}\n\n"
        f"\tannotation PBI_IncludeFutureArtifacts = False\n\n"
    )


def generate_model_tmdl(table_names, expression_source_name):
    refs = "\n".join([f"ref table {q(t)}" for t in table_names])
    return (
        f"model Model\n"
        f"\tculture: en-US\n"
        f"\tdefaultPowerBIDataSourceVersion: powerBI_V3\n"
        f"\tsourceQueryCulture: en-US\n"
        f"\tdataAccessOptions\n"
        f"\t\tlegacyRedirects\n"
        f"\t\treturnErrorValuesAsNull\n\n"
        f'annotation PBI_QueryOrder = ["{expression_source_name}"]\n\n'
        f"annotation __PBI_TimeIntelligenceEnabled = 0\n\n"
        f'annotation PBI_ProTooling = ["DirectLakeOnOneLakeInWeb","WebModelingEdit"]\n\n'
        f"{refs}\n"
    )


def generate_database_tmdl():
    return "database\n\tcompatibilityLevel: 1604\n"


def generate_pbism():
    return json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json",
        "version": "4.2",
        "settings": {},
    }, indent=2)


def generate_platform(display_name):
    return json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
        "metadata": {"type": "SemanticModel", "displayName": display_name},
        "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"},
    }, indent=2)


def encode(text):
    return base64.b64encode(text.encode("utf-8")).decode("utf-8")


# ──────────────────────────────────────────────────────────────────────────────
# MODEL ASSEMBLY
# ──────────────────────────────────────────────────────────────────────────────
def build_estate_tmdl(expression_source_name, directlake_url,
                      tables=ESTATE_TABLES, relationships=ESTATE_RELATIONSHIPS,
                      measures=ESTATE_MEASURES):
    """Return {tmdl_path: tmdl_text} for the full estate model. Pure text — the
    notebook base64-encodes each into a Fabric REST `parts` payload."""
    files = {}
    table_names = []

    for t in tables:
        columns_tmdl = "".join(
            generate_column_tmdl(c[0], c[1], c[2], c[3]) for c in t["columns"])
        files[f"definition/tables/{t['entity']}.tmdl"] = generate_table_tmdl(
            t["display"], t["entity"], columns_tmdl, expression_source_name)
        table_names.append(t["display"])

    rels_tmdl = generate_relationships_tmdl(relationships)
    if rels_tmdl:
        files["definition/relationships.tmdl"] = rels_tmdl

    files["definition/tables/_Metrics.tmdl"] = generate_metrics_table_tmdl(measures)
    table_names.append("_Metrics")

    files["definition/expressions.tmdl"] = generate_expressions_tmdl(
        expression_source_name, directlake_url)
    files["definition/model.tmdl"] = generate_model_tmdl(table_names, expression_source_name)
    files["definition/database.tmdl"] = generate_database_tmdl()
    return files, table_names


def build_estate_parts(display_name, expression_source_name, directlake_url, **kw):
    """Full Fabric REST `parts` list (base64) for the estate semantic model."""
    files, table_names = build_estate_tmdl(expression_source_name, directlake_url, **kw)
    parts = [{"path": p, "payload": encode(txt), "payloadType": "InlineBase64"}
             for p, txt in files.items()]
    parts.append({"path": "definition.pbism", "payload": encode(generate_pbism()),
                  "payloadType": "InlineBase64"})
    parts.append({"path": ".platform", "payload": encode(generate_platform(display_name)),
                  "payloadType": "InlineBase64"})
    return parts, table_names


## Step 1 — Read the Play 2 metadata tables

In [ ]:
def read_metadata_table(table_name):
    """Read a Play 2 Delta table to pandas, dropping any all-null (void) columns."""
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
    sdf = spark.sql(f"SELECT * FROM {METADATA_LAKEHOUSE}.dbo.{table_name}")
    from pyspark.sql.types import NullType
    drop = [f.name for f in sdf.schema.fields if isinstance(f.dataType, NullType)]
    if drop:
        sdf = sdf.drop(*drop)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    return sdf.toPandas()

raw_ds      = read_metadata_table("tableau_datasources").to_dict("records")
raw_fields  = read_metadata_table("tableau_fields").to_dict("records")
raw_lineage = read_metadata_table("tableau_lineage").to_dict("records")
raw_wb      = read_metadata_table("tableau_workbooks").to_dict("records")

print(f"✓ datasources: {len(raw_ds)}")
print(f"✓ fields:      {len(raw_fields)}")
print(f"✓ lineage:     {len(raw_lineage)}")
print(f"✓ workbooks:   {len(raw_wb)}")


## Step 2 — Derive the tidy `mig_*` assessment tables

Boolean/numeric assessment flags (`is_reused`, `is_refresh_stale`, `is_flat_file`,
`is_calculated`, `is_content_stale`, …) are computed here so the KPI DAX stays simple
and unambiguous — no delimited-string parsing in DAX. Connections come from the
**lineage** table at the real connection-instance grain (with a fallback to the
datasource's connection-type set when table-level lineage is absent).

In [ ]:
derived = {
    "mig_datasources": build_mig_datasources(raw_ds, stale_days=STALE_DAYS),
    "mig_fields":      build_mig_fields(raw_fields),
    "mig_connections": build_mig_connections(raw_lineage, raw_ds, flat_file_types=FLAT_FILE_TYPES),
    "mig_workbooks":   build_mig_workbooks(raw_wb, stale_days=STALE_DAYS),
    "mig_dim_project": build_dim_project(raw_ds, raw_wb),
    "mig_dim_owner":   build_dim_owner(raw_ds, raw_wb),
}
for name, rows in derived.items():
    print(f"  {name:18s} {len(rows):6d} rows")


## Step 3 — Write derived tables to Delta + assert the schema contract

Each table is written with an **explicit Spark schema** (string / boolean / bigint) so
the physical Delta types match what the TMDL generator expects. We then read the written
schema back and assert it against `expected_delta_contract()` — failing fast here
prevents the DirectLake binding error that a silent type drift would otherwise cause.

In [ ]:
_SPARK_TYPE = {"string": StringType(), "boolean": BooleanType(), "int64": LongType(), "dateTime": TimestampType()}

def _spark_schema(entity):
    cols = next(t["columns"] for t in ESTATE_TABLES if t["entity"] == entity)
    return StructType([StructField(c[0], _SPARK_TYPE[c[1]], True) for c in cols]), [c[0] for c in cols]

def write_delta(entity, rows):
    schema, order = _spark_schema(entity)
    tuples = [tuple(r.get(col) for col in order) for r in rows]
    sdf = spark.createDataFrame(tuples, schema)
    (sdf.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{METADATA_LAKEHOUSE}.{entity}"))
    return sdf.count()

for entity, rows in derived.items():
    n = write_delta(entity, rows)
    print(f"  ✓ wrote {entity:18s} ({n} rows)")

# ── Assert physical Delta schema matches the TMDL contract ──────────────────
problems = []
for entity in derived:
    actual = [(f.name, f.dataType.simpleString())
              for f in spark.table(f"{METADATA_LAKEHOUSE}.{entity}").schema.fields]
    problems += check_delta_schema(entity, actual)
if problems:
    raise Exception("Delta schema contract violations:\n  " + "\n  ".join(problems))
print("\n✓ All derived tables match the DirectLake schema contract")


## Step 4 — Generate the estate semantic model (TMDL)

In [ ]:
parts, table_names = build_estate_parts(
    MODEL_DISPLAY_NAME, EXPRESSION_SOURCE_NAME, DIRECTLAKE_URL)
print(f"✓ Tables:   {table_names}")
print(f"✓ Measures: {len(ESTATE_MEASURES)}")
print(f"✓ TMDL parts: {len(parts)}")


## Step 5 — Deploy + refresh via the Fabric REST API

In [ ]:
def get_existing_models():
    r = requests.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels", headers=HEADERS)
    r.raise_for_status()
    return {m["displayName"]: m["id"] for m in r.json().get("value", [])}

def deploy_semantic_model(display_name, parts, existing_id=None):
    definition = {"format": "TMDL", "parts": parts}
    if existing_id and OVERWRITE:
        resp = requests.post(
            f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels/{existing_id}/updateDefinition",
            headers=HEADERS, json={"definition": definition})
    else:
        resp = requests.post(
            f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items", headers=HEADERS,
            json={"displayName": display_name, "type": "SemanticModel", "definition": definition})
    if resp.status_code == 202:
        op = resp.headers.get("Location")
        wait = int(resp.headers.get("Retry-After", 20))
        for _ in range(30):
            time.sleep(wait)
            poll = requests.get(op, headers=HEADERS); poll.raise_for_status()
            res = poll.json(); status = res.get("status")
            if status == "Succeeded":
                return res
            if status in ("Failed", "Undefined"):
                raise Exception(f"Async operation failed: {res}")
            wait = int(poll.headers.get("Retry-After", wait))
        raise Exception("Async operation did not complete in time")
    resp.raise_for_status()
    return resp.json()

def refresh_model(model_id):
    url = f"https://api.powerbi.com/v1.0/myorg/datasets/{model_id}/refreshes"
    r = requests.post(url, headers=HEADERS, json={"type": "full", "commitMode": "transactional"})
    if r.status_code not in (200, 202):
        raise Exception(f"refresh POST {r.status_code}: {r.text[:200]}")
    for _ in range(30):
        time.sleep(8)
        top = requests.get(url + "?$top=1", headers=HEADERS).json().get("value", [])
        if not top:
            continue
        if top[0].get("status") == "Completed":
            return
        if top[0].get("status") == "Failed":
            raise Exception(f"refresh failed: {json.dumps(top[0])[:300]}")
    raise Exception("refresh did not complete in time")

existing = get_existing_models()
existing_id = existing.get(MODEL_DISPLAY_NAME)
if existing_id and not OVERWRITE:
    print(f"⚠ '{MODEL_DISPLAY_NAME}' already exists and OVERWRITE=False — skipping")
else:
    deploy_semantic_model(MODEL_DISPLAY_NAME, parts, existing_id)
    action = "updated" if existing_id else "created"
    print(f"✓ Semantic model {action}: '{MODEL_DISPLAY_NAME}'")
    model_id = existing_id or get_existing_models().get(MODEL_DISPLAY_NAME)
    if model_id:
        try:
            refresh_model(model_id)
            print("✓ DirectLake refreshed — model is query-ready")
        except Exception as e:
            print(f"⚠ Deployed, but refresh failed (refresh manually in Fabric): {e}")


## Step 6 — Verify

Confirms the model is present and prints the KPI catalog. Then open the dashboard
template in `Play5/dashboard/` (or Fabric → the model → **Auto-create report**) and
point it at **{MODEL_DISPLAY_NAME}**.

In [ ]:
r = requests.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels", headers=HEADERS)
r.raise_for_status()
names = [m["displayName"] for m in r.json().get("value", [])]
print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"  Model deployed: {'✓' if MODEL_DISPLAY_NAME in names else '✗ NOT FOUND'} — {MODEL_DISPLAY_NAME}")
print(f"  Tables: {table_names}")
print("\n  Migration KPIs (_Metrics):")
for nm, dax, fmt in ESTATE_MEASURES:
    print(f"    • {nm}")
print(f"\n  ✓ DirectLake → {METADATA_LAKEHOUSE}")
print(f"  ✓ Connect Play5/dashboard/ to '{MODEL_DISPLAY_NAME}'")
